# The real thing: cosmic web from real N-body data

No more toy fields. This notebook runs the exact pipeline we built — T-web classification and a 3D U-Net — on **real cosmological N-body data**, downloaded over plain HTTP (no Globus needed).

The data is from the **CAMELS Multifield Dataset** (same group behind QUIJOTE): gravity-only N-body simulations of dark matter, released as 3D total-matter density grids. We use the CV set — 27 independent realizations of the same fiducial cosmology on a periodic 25 Mpc/h box, 128³ voxels, at z=0. Real halos, real filaments, real voids.

The flow: download → map the real web with the T-web → train the 3D U-Net to recover that web from a sparse, galaxy-like tracer sampling → look at what it gets right. Run top to bottom. The download is ~216 MB; training is a couple of minutes on a multi-core CPU and seconds on a GPU (auto-detected).

## 0. Setup and download the real data

Pulls one CMD file (N-body total-matter density, CV set) straight from the CAMELS public server. Cached after the first run.

In [ ]:
# torch already in the env; no-op unless missing
!pip install torch

In [ ]:
import os, time, urllib.request, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from numpy.fft import fftn, ifftn, fftfreq
torch.manual_seed(0); np.random.seed(0); torch.set_num_threads(os.cpu_count() or 1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- config ----
URL   = "https://users.flatironinstitute.org/~camels/CMD/3D_grids/data/Nbody/Grids_Mtot_Nbody_IllustrisTNG_CV_128_z=0.0.npy"
FNAME = "cmd_cv128.npy"      # ~216 MB, 27 realizations of a 128^3 density grid
L         = 25.0            # box size of the CAMELS grids, Mpc/h
R_smooth  = 1.0            # T-web smoothing, Mpc/h (small box -> ~1 Mpc/h)
lam_th    = 0.3
nbar      = 3.0            # mean tracers per voxel for the sparse "galaxy" sampling
patch, stride = 32, 32     # 3D sub-cube size / spacing (32 -> 64 cubes per 128^3 grid)
f3d, epochs   = 12, 12
TRAIN_IDS = [0, 1, 2]      # which realizations to train on (more = better; up to ~25)
VAL_IDS   = [3, 4]
CLASS_NAMES  = ['void','sheet','filament','knot']
CLASS_COLORS = ['#08306b','#6baed6','#fd8d3c','#cb181d']
CMAP = ListedColormap(CLASS_COLORS); NORM = BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)

if not os.path.exists(FNAME):
    print("downloading ~216 MB from CAMELS public server ...")
    urllib.request.urlretrieve(URL, FNAME)
print("torch", torch.__version__, "| device:", device, "| data:", FNAME,
      f"({os.path.getsize(FNAME)/1e6:.0f} MB)")

## 1. Map the real cosmic web

Load one realization, convert the mass density to overdensity (δ = ρ/⟨ρ⟩ − 1), and classify it with the same tidal-tensor T-web. The density slice shows the real web — thin filaments between dense halos; the classification picks out voids, sheets, filaments, and knots.

In [ ]:
def smooth(d,L,R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real
def tweb(d,L,lam):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T)>lam,-1).astype(np.int64)
def fracs(lab): f=np.bincount(lab.ravel(),minlength=4); return (100*f/f.sum()).round(1)

grids = np.load(FNAME, mmap_mode='r')         # (27, 128, 128, 128), don't load all into RAM
print("dataset:", grids.shape)
rho = np.array(grids[0], dtype=np.float64); delta = rho/rho.mean() - 1.0
labels0 = tweb(smooth(delta, L, R_smooth), L, lam_th)
print(f"real field: delta in [{delta.min():.2f}, {delta.max():.0f}]   web % {dict(zip(CLASS_NAMES, fracs(labels0)))}")

z=64
fig,ax=plt.subplots(1,2,figsize=(13,6))
im0=ax[0].imshow(np.log10(1+delta[z].T+1e-3),origin='lower',cmap='inferno',extent=[0,L,0,L],vmin=-1,vmax=2.5)
ax[0].set_title('REAL N-body density  log10(1+delta)'); ax[0].set_xlabel('Mpc/h'); ax[0].set_ylabel('Mpc/h'); plt.colorbar(im0,ax=ax[0],fraction=.046)
im1=ax[1].imshow(labels0[z].T,origin='lower',cmap=CMAP,norm=NORM,extent=[0,L,0,L]); ax[1].set_title('cosmic web (T-web)'); ax[1].set_xlabel('Mpc/h')
cb=plt.colorbar(im1,ax=ax[1],ticks=[0,1,2,3],fraction=.046); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## 2. Build the training data from real realizations

For each chosen realization: compute the web labels (target) and a sparse Poisson-tracer view of the density (input). Real densities span ~6 orders of magnitude, so we feed log10(1+counts) to keep the dynamic range sane. Each 128³ grid is cut into 3D sub-cubes.

In [ ]:
def prep(rho, seed):
    delta = rho/rho.mean() - 1.0
    lab = tweb(smooth(delta, L, R_smooth), L, lam_th)
    counts = np.random.default_rng(seed).poisson(nbar*np.clip(1+delta, 0, None))
    return np.log10(1.0+counts).astype(np.float32), lab
def patches(vol, lab, p, s):
    N=vol.shape[0]; X,Y=[],[]
    for i in range(0,N-p+1,s):
        for j in range(0,N-p+1,s):
            for k in range(0,N-p+1,s): X.append(vol[i:i+p,j:j+p,k:k+p]); Y.append(lab[i:i+p,j:j+p,k:k+p])
    return np.array(X), np.array(Y)

def build(ids):
    X,Y=[],[]
    for gi in ids:
        inp,lab = prep(np.array(grids[gi], dtype=np.float64), 100+gi)
        P,Q = patches(inp, lab, patch, stride); X.append(P); Y.append(Q)
    return np.concatenate(X), np.concatenate(Y)

t0=time.time()
Xtr,Ytr = build(TRAIN_IDS); Xva,Yva = build(VAL_IDS)
print(f"prepped {len(TRAIN_IDS)}+{len(VAL_IDS)} realizations in {time.time()-t0:.0f}s")
print(f"train cubes {Xtr.shape}  val {Xva.shape}  classmix% {fracs(Ytr)}")

## 3. Train the 3D U-Net on real data

The same 3D network as the toy notebook. Gentle (sqrt-inverse-frequency) class weights so the rare knot class is helped but not over-predicted, and we keep the best-epoch model.

In [ ]:
def double_conv(ci,co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True),
                         nn.Conv3d(co,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True))
class UNet3D(nn.Module):
    def __init__(self,f=12,nc=4):
        super().__init__(); self.e1=double_conv(1,f); self.e2=double_conv(f,2*f); self.b=double_conv(2*f,4*f)
        self.u2=nn.ConvTranspose3d(4*f,2*f,2,2); self.d2=double_conv(4*f,2*f)
        self.u1=nn.ConvTranspose3d(2*f,f,2,2); self.d1=double_conv(2*f,f); self.o=nn.Conv3d(f,nc,1); self.p=nn.MaxPool3d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.o(d1)
def per_class_f1(p,t,nc=4):
    o=[]
    for c in range(nc):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum(); o.append(2*tp/(2*tp+fp+fn+1e-9))
    return o

mu,sd = Xtr.mean(), Xtr.std()
Xt=torch.tensor(((Xtr-mu)/sd)[:,None]).to(device); Yt=torch.tensor(Ytr).to(device)
Xv=torch.tensor(((Xva-mu)/sd)[:,None]).to(device); Yv=torch.tensor(Yva).to(device)
fr=np.bincount(Ytr.ravel(),minlength=4); w=(1.0/np.sqrt(fr+1)); w=(w/w.sum()*4).astype(np.float32)
net=UNet3D(f3d).to(device); opt=torch.optim.Adam(net.parameters(),1e-3)
crit=nn.CrossEntropyLoss(weight=torch.tensor(w).to(device))
print(f"{sum(p.numel() for p in net.parameters())} params on {device}")
bs,n=4,len(Xt); best=(0.0,None,None); t0=time.time()
for ep in range(epochs):
    net.train(); perm=torch.randperm(n)
    for i in range(0,n,bs):
        idx=perm[i:i+bs]; opt.zero_grad(); crit(net(Xt[idx]),Yt[idx]).backward(); opt.step()
    net.eval()
    with torch.no_grad(): pv=net(Xv).argmax(1).cpu().numpy()
    acc=(pv==Yva).mean(); F=per_class_f1(pv,Yva)
    if acc>best[0]: best=(acc,F,pv)
    print(f"epoch {ep+1:2d}  valAcc {acc:.3f}   F1 void {F[0]:.2f} sheet {F[1]:.2f} filament {F[2]:.2f} knot {F[3]:.2f}   ({time.time()-t0:.0f}s)")
acc,F,pred = best; maj=fr.argmax()
print(f"\nBEST valAcc {acc:.3f}  (baseline {(Yva==maj).mean():.3f})   "
      f"F1 void {F[0]:.2f} sheet {F[1]:.2f} filament {F[2]:.2f} knot {F[3]:.2f}")

## 4. See it work on real data

A central slice through a validation sub-cube: real sparse tracers in, the real N-body web, and the U-Net's reconstruction (best-epoch model).

In [ ]:
zc=patch//2
scores=[min(np.bincount(Yva[i].ravel(),minlength=4)/Yva[i].size) for i in range(len(Yva))]
pi=int(np.argmax(scores))     # show a patch with all four classes present
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(Xva[pi,zc].T,origin='lower',cmap='inferno'); ax[0].set_title('REAL sparse tracers (slice of 3D cube)')
ax[1].imshow(Yva[pi,zc].T,origin='lower',cmap=CMAP,norm=NORM); ax[1].set_title('true web (real N-body)')
im=ax[2].imshow(pred[pi,zc].T,origin='lower',cmap=CMAP,norm=NORM); ax[2].set_title('3D U-Net prediction')
cb=plt.colorbar(im,ax=ax[2],ticks=[0,1,2,3],fraction=.046); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## What just happened, and where it goes

The full pipeline — generate/ingest a density field, classify the web, recover it from sparse tracers — now runs on **real cosmological N-body data**, no toy fields and no Globus. Performance is a bit below the toy numbers because real structure is genuinely harder: finer filaments, real halos, sharper contrasts.

Natural next moves, all unlocked from here:

- **Use more of the 27 realizations** (extend `TRAIN_IDS`) and a GPU — the single biggest lever on the rare-class (knot) F1.
- **Cross-check the voids** against the real `VIDE_Voids/` and `Disperse/` catalogs sitting next to this data on the CAMELS server — an independent check on what the network calls a void.
- **Vary cosmology.** The LH set spans 1,000 different (Ωm, σ8) universes; the same pipeline becomes a path to *parameter inference* — predicting the cosmology from the web itself.
- **Step up to QUIJOTE's 1 Gpc boxes** via Globus once you want the large-scale web, using this exact `np.load` → T-web → U-Net path.